In [1]:
import pandas as pd
from prophet import Prophet
from dotenv import load_dotenv
import plotly.express as px
import os
import utils
load_dotenv()

/home/nbillard/miniconda3/envs/conda-jedha/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
df_rte = pd.read_csv(os.getenv('DATA_STORAGE') + 'power_consumption_meteo_2021_2025_utc.csv')
df_rte.head()

,start_date,end_date,value,mean,min,max,median,std,q1,q3
0,2021-01-01 00:00:00,2021-01-01 01:00:00,63955.5,1.636,-2.5,9.9,1.0,2.849473,-0.575,2.975
1,2021-01-01 01:00:00,2021-01-01 02:00:00,62054.0,1.340,-3.5,10.2,1.0,3.125145,-1.500,3.300
2,2021-01-01 02:00:00,2021-01-01 03:00:00,59021.5,1.092,-3.1,10.0,0.8,3.180864,-2.000,2.875
3,2021-01-01 03:00:00,2021-01-01 04:00:00,57391.5,0.966,-2.8,9.2,0.5,3.160574,-1.775,2.875
4,2021-01-01 04:00:00,2021-01-01 05:00:00,57620.5,0.802,-2.8,9.6,0.0,3.094530,-1.675,2.650


In [3]:
dfp = utils.convert_df_to_prophet(df_rte)
dfp.head()

,ds,y,mean,min,max,median,std,q1,q3
0,2021-01-01 01:00:00,63955.5,1.636,-2.5,9.9,1.0,2.849473,-0.575,2.975
1,2021-01-01 02:00:00,62054.0,1.340,-3.5,10.2,1.0,3.125145,-1.500,3.300
2,2021-01-01 03:00:00,59021.5,1.092,-3.1,10.0,0.8,3.180864,-2.000,2.875
3,2021-01-01 04:00:00,57391.5,0.966,-2.8,9.2,0.5,3.160574,-1.775,2.875
4,2021-01-01 05:00:00,57620.5,0.802,-2.8,9.6,0.0,3.094530,-1.675,2.650


--------------------------------------------------------------------

In [ ]:
dataset = dfp.copy()
dataset = dataset[dataset['ds'] < '2025-10-01']
train_period = "30 days"
test_period = "1 day"
period = "53 hours" # to have differents dayofweek and hours
regressors = ['mean','min','max','std','q1','q3']
title = f"last_date: {dataset['ds'].max()} - train_period: {train_period} - test_period: {test_period} - period: {period} - regressors: {",".join(regressors)}"
train_test_sets = utils.create_train_test(dataset, train_period, test_period, period=period, n_iter=5)

In [5]:
results_reg, metrics_reg = utils.train_predict(train_test_sets, regressors=regressors)

15:56:14 - cmdstanpy - INFO - Chain [1] start processing
15:56:14 - cmdstanpy - INFO - Chain [1] done processing
15:56:15 - cmdstanpy - INFO - Chain [1] start processing
15:56:15 - cmdstanpy - INFO - Chain [1] done processing


SET 0
train from 2025-08-22 03:00:00 to 2025-09-21 02:00:00
test from 2025-09-21 03:00:00 to 2025-09-22 02:00:00
Train MAE: 1056.88, Test MAE: 1384.71
Train MAPE: 2.62%, Test MAPE: 3.77%
--------------------------------------------------
SET 1
train from 2025-08-24 08:00:00 to 2025-09-23 07:00:00
test from 2025-09-23 08:00:00 to 2025-09-24 07:00:00


15:56:15 - cmdstanpy - INFO - Chain [1] start processing
15:56:15 - cmdstanpy - INFO - Chain [1] done processing
15:56:15 - cmdstanpy - INFO - Chain [1] start processing
15:56:15 - cmdstanpy - INFO - Chain [1] done processing


Train MAE: 1016.26, Test MAE: 1660.84
Train MAPE: 2.51%, Test MAPE: 3.56%
--------------------------------------------------
SET 2
train from 2025-08-26 13:00:00 to 2025-09-25 12:00:00
test from 2025-09-25 13:00:00 to 2025-09-26 12:00:00
Train MAE: 1023.10, Test MAE: 1252.48
Train MAPE: 2.52%, Test MAPE: 2.52%
--------------------------------------------------
SET 3
train from 2025-08-28 18:00:00 to 2025-09-27 17:00:00
test from 2025-09-27 18:00:00 to 2025-09-28 17:00:00


15:56:15 - cmdstanpy - INFO - Chain [1] start processing
15:56:15 - cmdstanpy - INFO - Chain [1] done processing


Train MAE: 1062.41, Test MAE: 2889.50
Train MAPE: 2.60%, Test MAPE: 7.21%
--------------------------------------------------
SET 4
train from 2025-08-30 23:00:00 to 2025-09-29 22:00:00
test from 2025-09-29 23:00:00 to 2025-09-30 22:00:00
Train MAE: 1142.41, Test MAE: 1762.29
Train MAPE: 2.78%, Test MAPE: 3.95%
--------------------------------------------------
TRAIN MAPE MEAN: 2.61% -             TEST MAPE MEAN: 4.20%


In [6]:
utils.log_metrics(metrics_reg, title)

### Modèle de base sans regressor

In [5]:
results, metrics = utils.train_predict(train_test_sets)

15:09:32 - cmdstanpy - INFO - Chain [1] start processing
15:09:32 - cmdstanpy - INFO - Chain [1] done processing
15:09:32 - cmdstanpy - INFO - Chain [1] start processing
15:09:32 - cmdstanpy - INFO - Chain [1] done processing


SET 0
train from 2025-08-22 03:00:00 to 2025-09-21 02:00:00
test from 2025-09-21 03:00:00 to 2025-09-22 02:00:00
Train MAE: 1060.76, Test MAE: 1881.95
Train MAPE: 2.63%, Test MAPE: 5.05%
--------------------------------------------------
SET 1
train from 2025-08-24 08:00:00 to 2025-09-23 07:00:00
test from 2025-09-23 08:00:00 to 2025-09-24 07:00:00


15:09:32 - cmdstanpy - INFO - Chain [1] start processing
15:09:32 - cmdstanpy - INFO - Chain [1] done processing
15:09:32 - cmdstanpy - INFO - Chain [1] start processing
15:09:32 - cmdstanpy - INFO - Chain [1] done processing


Train MAE: 1038.13, Test MAE: 1338.96
Train MAPE: 2.56%, Test MAPE: 2.87%
--------------------------------------------------
SET 2
train from 2025-08-26 13:00:00 to 2025-09-25 12:00:00
test from 2025-09-25 13:00:00 to 2025-09-26 12:00:00
Train MAE: 1054.40, Test MAE: 1464.78
Train MAPE: 2.59%, Test MAPE: 2.92%
--------------------------------------------------
SET 3
train from 2025-08-28 18:00:00 to 2025-09-27 17:00:00
test from 2025-09-27 18:00:00 to 2025-09-28 17:00:00


15:09:32 - cmdstanpy - INFO - Chain [1] start processing
15:09:32 - cmdstanpy - INFO - Chain [1] done processing


Train MAE: 1073.17, Test MAE: 2890.09
Train MAPE: 2.63%, Test MAPE: 7.19%
--------------------------------------------------
SET 4
train from 2025-08-30 23:00:00 to 2025-09-29 22:00:00
test from 2025-09-29 23:00:00 to 2025-09-30 22:00:00
Train MAE: 1157.83, Test MAE: 2005.55
Train MAPE: 2.82%, Test MAPE: 4.49%
--------------------------------------------------
TRAIN MAPE MEAN: 2.64% -             TEST MAPE MEAN: 4.50%


In [6]:
utils.log_metrics(metrics, f"BASE sans regressor - {title}")

In [23]:
# chart predict
def chart_predictions(df, results, results_reg, title, display_period="30 days"):

    display_delta = pd.Timedelta(display_period)
    print(display_delta)
    last_date = df['ds'].max()
    display_start = last_date - display_delta
    display_mask = df['ds'] > display_start
    print(last_date, display_start)

    fig = px.line(df[display_mask], x='ds', y='y', range_y=[0, None], title=title)

    i = 1
    for res in results:
        train_pred, test_pred = res
        fig.add_trace(px.scatter(test_pred, x='ds', y='yhat').data[0])
        fig.data[i].marker = dict(color='red', size=3)
        i = i+1

    for res in results_reg:
        train_pred, test_pred = res
        fig.add_trace(px.scatter(test_pred, x='ds', y='yhat').data[0])
        fig.data[i].marker = dict(color='green', size=3)
        i = i+1
        
    fig.show()

In [25]:
chart_predictions(dataset, results, results_reg, title, "15 days")

15 days 00:00:00
2025-09-30 23:00:00 2025-09-15 23:00:00
